In [ ]:
#pip install duckdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 6.7 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


Hey!
In this file you can find a litlle show case of my SQL skils.Data here are not very complicated and already has been cleared so waht you see below its very quick and small Exploratory Data Analysis. 

In [3]:
import pandas as pd
import duckdb

In [16]:
income=pd.read_csv("Data_for_Looker/Income.csv")
Spendings=pd.read_csv("Data_for_Looker/Spendings.csv")

income=income.rename(columns={"Data of Transaction":"Date_of_Transaction"})
Spendings=Spendings.rename(columns={"Data of Transaction":"Date_of_Transaction"})

In [25]:
Cleaned_income=duckdb.query("""
SELECT Date_of_Transaction, Amt, Main_Category,Sub_Category
FROM income
""")


Cleaned_income


┌─────────────────────┬─────────┬───────────────┬────────────────────┐
│ Date_of_Transaction │   Amt   │ Main_Category │    Sub_Category    │
│       varchar       │ double  │    varchar    │      varchar       │
├─────────────────────┼─────────┼───────────────┼────────────────────┤
│ 2026-04-10          │ 4430.47 │ Income        │ Salary             │
│ 2026-04-30          │    65.0 │ Income        │ External Transfers │
│ 2026-03-10          │ 4430.47 │ Income        │ Salary             │
│ 2026-03-31          │    80.0 │ Income        │ External Transfers │
│ 2026-02-06          │   -50.0 │ Income        │ BLIK               │
│ 2026-02-10          │ 4430.47 │ Income        │ Salary             │
│ 2026-01-06          │   100.0 │ Income        │ BLIK               │
│ 2026-01-10          │ 4430.47 │ Income        │ Salary             │
│ 2026-01-30          │    50.0 │ Income        │ External Transfers │
└─────────────────────┴─────────┴───────────────┴────────────────────┘

In [24]:
Cleaned_Spendings=duckdb.query("""
SELECT Date_of_Transaction, Amt, Main_Category,Sub_Category
FROM Spendings
""")

Cleaned_Spendings


┌─────────────────────┬────────┬────────────────┬──────────────┐
│ Date_of_Transaction │  Amt   │ Main_Category  │ Sub_Category │
│       varchar       │ double │    varchar     │   varchar    │
├─────────────────────┼────────┼────────────────┼──────────────┤
│ 2026-04-02          │ 3600.0 │ Fixed Costs    │ Rent         │
│ 2026-04-02          │   36.0 │ Smart saver    │ Smart saver  │
│ 2026-04-02          │   58.9 │ Daily Purchase │ Biednronka   │
│ 2026-04-03          │   13.5 │ Daily Purchase │ Zabka        │
│ 2026-04-03          │   0.13 │ Smart saver    │ Smart saver  │
│ 2026-04-04          │  125.4 │ Transport      │ Fuel         │
│ 2026-04-04          │   30.5 │ Eating out     │ KFC          │
│ 2026-04-05          │ 110.65 │ Fixed Costs    │ Debt pay     │
│ 2026-04-05          │   1.11 │ Smart saver    │ Smart saver  │
│ 2026-04-05          │   95.2 │ Daily Purchase │ Lidl         │
│     ·               │     ·  │     ·          │  ·           │
│     ·               │  

In [ ]:
Income_MainCat=duckdb.query("""
Select Main_Category,Round(Sum(Amt),2) as Amount
From Cleaned_income
Group by Main_Category 
Order by Amount Desc
""")

Income_SubCat=duckdb.query("""
Select Sub_Category,Round(Sum(Amt),2) as Amount
From Cleaned_income
Group by Sub_Category 
Order by Amount Desc
""")

print(Income_MainCat)
print(Income_SubCat)

┌───────────────┬──────────┐
│ Main_Category │  Amount  │
│    varchar    │  double  │
├───────────────┼──────────┤
│ Income        │ 17966.88 │
└───────────────┴──────────┘

┌────────────────────┬──────────┐
│    Sub_Category    │  Amount  │
│      varchar       │  double  │
├────────────────────┼──────────┤
│ Salary             │ 17721.88 │
│ External Transfers │    195.0 │
│ BLIK               │     50.0 │
└────────────────────┴──────────┘



In [ ]:
Spendings_MainCat=duckdb.query("""
Select Main_Category,Round(Sum(Amt),2) as Amount
From Cleaned_Spendings
Group by Main_Category
Order By Amount DESC
""")


Spendings_SubCat=duckdb.query("""
Select Sub_Category,Round(Sum(Amt),2) as Amount
From Cleaned_Spendings
Group by Sub_Category
Order By Amount DESC
""")

print(Spendings_MainCat)
print(Spendings_SubCat)

In [44]:
Limited_income=duckdb.query("""
Select Sub_Category,Round(Sum(Amt),2) as Amount
From Cleaned_income
Group by Sub_Category
Having Amount>1000
Order By Amount DESC
""")

print(Limited_income)

Limited_spendings=duckdb.query("""
Select*
From(Select Sub_Category,Round(Sum(Amt),2) as Amount
     From Cleaned_Spendings
     Group by Sub_Category )
Where Amount>1000
Order by Amount DESC
""")

print(Limited_spendings)


┌──────────────┬──────────┐
│ Sub_Category │  Amount  │
│   varchar    │  double  │
├──────────────┼──────────┤
│ Salary       │ 17721.88 │
└──────────────┴──────────┘

┌──────────────┬─────────┐
│ Sub_Category │ Amount  │
│   varchar    │ double  │
├──────────────┼─────────┤
│ Rent         │ 14400.0 │
│ Fuel         │  6351.2 │
│ Lidl         │ 2453.05 │
│ Biednronka   │  2286.2 │
└──────────────┴─────────┘



In [ ]:
Unioned_data=duckdb.query("""
Select *
From(From Cleaned_Spendings 
    Union All
    Select*
    From Cleaned_income)
Order By Date_of_Transaction ASC
""")

Unioned_data

┌─────────────────────┬────────┬────────────────┬────────────────────┐
│ Date_of_Transaction │  Amt   │ Main_Category  │    Sub_Category    │
│       varchar       │ double │    varchar     │      varchar       │
├─────────────────────┼────────┼────────────────┼────────────────────┤
│ 2026-01-02          │ 3600.0 │ Fixed Costs    │ Rent               │
│ 2026-01-02          │   36.0 │ Smart saver    │ Smart saver        │
│ 2026-01-02          │   14.5 │ Daily Purchase │ Zabka              │
│ 2026-01-03          │   52.3 │ Daily Purchase │ Biednronka         │
│ 2026-01-03          │   0.52 │ Smart saver    │ Smart saver        │
│ 2026-01-03          │    9.5 │ Eating out     │ KAWA               │
│ 2026-01-03          │   88.9 │ Transport      │ Fuel               │
│ 2026-01-04          │  118.4 │ Transport      │ Fuel               │
│ 2026-01-04          │   1.18 │ Smart saver    │ Smart saver        │
│ 2026-01-04          │   73.2 │ Daily Purchase │ Biednronka         │
│     

In [ ]:
#On this data join not realy make seans, maybe only to check how much was spend on the day when funds were recvived

Joined_data=duckdb.query(""" 
Select i.Date_of_Transaction,i.Sub_Category as Sub_Category_Income, i.Amt as Income,s.Main_Category as Main_Category_Spendings, s.Sub_Category as Sub_Category_Spendings, s.Amt as Spendings
From Cleaned_income as i Left Join Cleaned_Spendings s on s.Date_of_Transaction=i.Date_of_Transaction
Order By i.Date_of_Transaction ASC
""")

BinderException: Binder Error: column "Sub_Category" must appear in the GROUP BY clause or must be part of an aggregate function.
Either add it to the GROUP BY list, or use "ANY_VALUE(Sub_Category)" if the exact value of "Sub_Category" is not important.

In [61]:
Joined_data

┌─────────────────────┬─────────────────────┬─────────┬─────────────────────────┬────────────────────────┬───────────┐
│ Date_of_Transaction │ Sub_Category_Income │ Income  │ Main_Category_Spendings │ Sub_Category_Spendings │ Spendings │
│       varchar       │       varchar       │ double  │         varchar         │        varchar         │  double   │
├─────────────────────┼─────────────────────┼─────────┼─────────────────────────┼────────────────────────┼───────────┤
│ 2026-01-06          │ BLIK                │   100.0 │ Daily Purchase          │ Biednronka             │     145.6 │
│ 2026-01-10          │ Salary              │ 4430.47 │ Daily Purchase          │ Lidl                   │     89.45 │
│ 2026-01-10          │ Salary              │ 4430.47 │ Daily Purchase          │ Zabka                  │      11.2 │
│ 2026-01-30          │ External Transfers  │    50.0 │ Transport               │ Fuel                   │     150.5 │
│ 2026-01-30          │ External Transfers  │   